# MeAJOR dataset — Train baseline model
This notebook trains the project's baseline text classifier (TF-IDF + LogisticRegression) on the MeAJOR dataset located in `data/raw/meajor_cleaned_preprocessed.csv` and prints evaluation statistics (accuracy, precision, recall, F1, confusion matrix, per-class false negative rates).

In [5]:
# Standard imports and project path setup
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Ensure repo root is on sys.path so imports from src/ work when running the notebook
repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

print('Repository root:', repo_root)

Repository root: C:\Users\brian\phishing-investigator


In [6]:
# Project imports (reuse the same model and loader used by the project)
from src.ingestion.major_loader import load_major_training_dataset
from src.models.train import build_baseline_model

# Dataset path (update if you store it elsewhere)
DATASET_PATH = repo_root / 'data' / 'raw' / 'meajor_cleaned_preprocessed.csv'
print('Dataset path:', DATASET_PATH)

# Load dataset using the MeAJOR loader and build text from sender_domain, subject, and body
data = load_major_training_dataset(str(DATASET_PATH))
print('Loaded records:', len(data))
print('Columns available for training:', list(data.columns))
display(data.head())
print('Label distribution:')
display(data['label'].value_counts())

Dataset path: C:\Users\brian\phishing-investigator\data\raw\meajor_cleaned_preprocessed.csv
Dropped 1 rows with missing labels
Loaded records: 108684
Columns available for training: ['sender_domain', 'subject', 'body', 'text', 'label', 'source_dataset', 'source_type']


,sender_domain,subject,body,text,label,source_dataset,source_type
0,enron.com,[organization] failover plan.,"hi [name], tonight we are rolling out a new re...",enron.com [organization] failover plan. hi [na...,0,meajor_cleaned_preprocessed.csv,meajor_csv
1,enron.com,re: intranet site,"[name] r these new? intranet site [name], we n...",enron.com re: intranet site [name] r these new...,0,meajor_cleaned_preprocessed.csv,meajor_csv
2,enron.com,fw: [organization] company information,"[name]/[name], we are currently trading under ...",enron.com fw: [organization] company informati...,0,meajor_cleaned_preprocessed.csv,meajor_csv
3,enron.com,new master physical,[name] and [name] - attached is a worksheet fo...,enron.com new master physical [name] and [name...,0,meajor_cleaned_preprocessed.csv,meajor_csv
4,enron.com,fw: [organization]/mirant gisb,fyi. below is a copy of my communication with ...,enron.com fw: [organization]/mirant gisb fyi. ...,0,meajor_cleaned_preprocessed.csv,meajor_csv


Label distribution:


label
0    60650
1    48034
Name: count, dtype: int64

In [7]:
# Split, build, train
X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = build_baseline_model()
model.fit(X_train, y_train)
preds = model.predict(X_test)
labels = list(model.named_steps['classifier'].classes_)
print('Trained baseline model with labels:', labels)

Trained baseline model with labels: [np.int64(0), np.int64(1)]


In [9]:
# Compute and display evaluation metrics including per-class false negative rate (FNR)
acc = accuracy_score(y_test, preds)
prec = precision_score(y_test, preds, average='weighted', zero_division=0)
rec = recall_score(y_test, preds, average='weighted', zero_division=0)
f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
cm = confusion_matrix(y_test, preds, labels=labels)

print(f'Accuracy: {acc:.4f}')
print(f'Precision (weighted): {prec:.4f}')
print(f'Recall (weighted): {rec:.4f}')
print(f'F1 (weighted): {f1:.4f}')

print('Confusion matrix (rows=true, cols=predicted)')
print(labels)
print(cm)

# Per-class metrics + False Negative Rate (FNR = FN / (TP+FN))
rows = []
for i, label in enumerate(labels):
    tp = int(cm[i, i])
    fn = int(cm[i, :].sum() - tp)
    fp = int(cm[:, i].sum() - tp)
    tn = int(cm.sum() - (tp + fn + fp))
    support = int(cm[i, :].sum())
    fnr = fn / support if support > 0 else np.nan
    rows.append({
        'label': label,
        'support': support,
        'tp': tp,
        'fn': fn,
        'fp': fp,
        'tn': tn,
        'precision': precision_score(y_test, preds, labels=[label], average='weighted', zero_division=0),
        'recall': recall_score(y_test, preds, labels=[label], average='weighted', zero_division=0),
        'f1': f1_score(y_test, preds, labels=[label], average='weighted', zero_division=0),
        'fnr': fnr,
    })

metrics_df = pd.DataFrame(rows).set_index('label')
display(metrics_df)

print('Classification report:')
print(classification_report(y_test, preds, zero_division=0))

Accuracy: 0.9790
Precision (weighted): 0.9790
Recall (weighted): 0.9790
F1 (weighted): 0.9790
Confusion matrix (rows=true, cols=predicted)
[np.int64(0), np.int64(1)]
[[11888   242]
 [  215  9392]]


,support,tp,fn,fp,tn,precision,recall,f1,fnr
label,,,,,,,,,
0,12130,11888,242,215,9392,0.982236,0.980049,0.981141,0.019951
1,9607,9392,215,242,11888,0.974881,0.977620,0.976249,0.022380


Classification report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     12130
           1       0.97      0.98      0.98      9607

    accuracy                           0.98     21737
   macro avg       0.98      0.98      0.98     21737
weighted avg       0.98      0.98      0.98     21737



In [ ]:
# Optionally save the trained model so you can load it later (uncomment to enable)
# import joblib
# out_path = repo_root / 'models' / 'phishing_classifier_meajor.pkl'
# out_path.parent.mkdir(parents=True, exist_ok=True)
# joblib.dump(model, out_path)
# print('Model saved to', out_path)

print('Notebook cells complete. Run all cells to train and evaluate the model.')